**Assistente acadêmico UFRPE com RAG**

Experimento de matrícula e trancamento na graduação presencial. O notebook executa a mesma extração, recuperação e geração usadas pela aplicação. A base foi conferida em 23/11/2026. Os rótulos de avaliação ainda precisam de revisão independente.

Execute primeiro a instalação descrita no README e selecione o Python desse ambiente como kernel. A primeira consulta baixa os modelos públicos. As próximas consultas podem usar o cache.

In [1]:
from pathlib import Path
import os, sys, csv, json

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "rag_ufrpe").is_dir(), "Abra este notebook dentro da pasta do repositório."
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
print("Projeto carregado:", ROOT.name)

Projeto carregado: assistente-ufrpe-rag


A coleção tem dois documentos oficiais. O manifesto registra origem e integridade dos arquivos. A extração mantém os parágrafos dos artigos e as datas dos eventos do calendário.

In [2]:
from rag_ufrpe.corpus import sources, build_corpus
for source in sources():
    print(source["title"])
    print(source["page_url"])
chunks = build_corpus()
print("Total de trechos:", len(chunks))
print("Exemplo:", next(c["text"] for c in chunks if c["id"] == "cal2026-107"))

Regulamento Geral da Graduação - Resolução 744/2024
https://seg.ufrpe.br/content/ufrpe-tem-novas-regras-para-solicitacao-de-colacao-de-grau-antecipada
Calendário da Graduação presencial 2026 - Resolução 1004/2026
https://seg.ufrpe.br/content/res-no-10042026


Total de trechos: 63
Exemplo: Agosto/2026 - 10 a 28. Período de solicitação, no SIGAA, de trancamento de matrícula em turmas para o período letivo de 2026.2


A busca combina embeddings multilíngues com TF-IDF. A seleção considera o semestre para perguntas sobre prazos. Cada trecho mantém título, páginas e endereço da fonte.

In [3]:
from rag_ufrpe.retrieval import Retriever
retriever = Retriever(chunks)
question = "Qual o prazo de trancamento de disciplina em 2026.2?"
retrieval = retriever.search(question)
for i, source in enumerate(retrieval["sources"], start=1):
    print(f"[{i}] {source["id"]} — páginas {source["pages"]}")
    print(source["text"], "\n")

[1] cal2026-107 — páginas [12]
Agosto/2026 - 10 a 28. Período de solicitação, no SIGAA, de trancamento de matrícula em turmas para o período letivo de 2026.2 



Perguntas de regra usam o modelo local, que recebe a pergunta e o primeiro trecho do ranking e devolve resposta e fonte em JSON. Se o trecho não responde, o modelo deve abster-se. O programa valida o formato, bloqueia números e vocabulário ausentes da fonte e exibe a citação. Perguntas de data são respondidas diretamente a partir do evento do calendário, sem modelo. As demais fontes ficam disponíveis para leitura. A avaliação do apoio de cada afirmação exige revisão humana.

In [4]:
from rag_ufrpe.generation import answer
# Pergunta de data: a resposta é montada a partir do evento, sem modelo.
result = answer(retrieval, question, backend="local")
print("Pergunta:", question)
print("Resposta:", result["answer"])
print("Modo:", result["mode"])
assert result["mode"] == "calendar"

# Pergunta de regra: o modelo local elabora a resposta e o programa a confere.
question = "Em quanto tempo o trancamento de uma disciplina é efetivado?"
result = answer(retriever.search(question), question, backend="local")
print("\nPergunta:", question)
print("Resposta:", result["answer"])
print("Modo:", result["mode"])
assert result["mode"] == "generated", "Confira a configuração do gerador e as fontes retornadas."

Pergunta: Qual o prazo de trancamento de disciplina em 2026.2?
Resposta: Período de solicitação, no SIGAA, de trancamento de matrícula em turmas para o período letivo de 2026.2: Agosto/2026 - 10 a 28. [1]
Modo: calendar



Pergunta: Em quanto tempo o trancamento de uma disciplina é efetivado?
Resposta: O trancamento de uma disciplina é efetivado 1 (um) dia após a solicitação. [1]
Modo: generated


Perguntas sobre datas usam os eventos do semestre solicitado. Pedidos sem semestre ou sobre um período ausente exigem esclarecimento ou abstenção.

In [5]:
examples = ["Qual o prazo de trancamento em 2026.2?", "Qual o prazo de matrícula?", "Qual o prazo de matrícula em 2027.1?", "Qual é a capital da França?"]
for question in examples:
    found = retriever.search(question)
    print(question, "=>", found["status"])
    print(found["sources"][0]["text"] if found["sources"] else found["message"])

Qual o prazo de trancamento em 2026.2? => ok
Agosto/2026 - 10 a 28. Período de solicitação, no SIGAA, de trancamento de matrícula em turmas para o período letivo de 2026.2
Qual o prazo de matrícula? => clarify
Informe o semestre, 2026.1 ou 2026.2, para consultar uma data.
Qual o prazo de matrícula em 2027.1? => unsupported_period
A base de prazos cobre somente 2026.1 e 2026.2. Consulte o calendário oficial do período solicitado.
Qual é a capital da França? => out_of_scope
Este assistente consulta matrícula e trancamento na graduação presencial da UFRPE.


A amostra contém 10 casos de desenvolvimento, 10 de teste e 20 de paráfrases. O conjunto de paráfrases foi montado depois de sondar falhas da busca, portanto serve de diagnóstico e não de teste independente. A avaliação abaixo mede a recuperação e os filtros, sem medir a correção do texto gerado.

In [6]:
from rag_ufrpe.evaluation import evaluate_retrieval
for split in ["dev", "test", "parafrase"]:
    print(json.dumps(evaluate_retrieval(split, retriever), ensure_ascii=False, indent=2))

{
  "date": "2026-11-25",
  "split": "dev",
  "k": 3,
  "cases": 10,
  "answerable_cases": 6,
  "abstention_cases": 4,
  "macro_precision_at_3": 0.3333,
  "macro_precision_returned": 0.6667,
  "macro_recall_at_3": 1.0,
  "macro_f1_at_3": 0.5,
  "hit_at_3": 1.0,
  "hit_at_1": 1.0,
  "mrr_at_3": 1.0,
  "answerable_status_accuracy": 1.0,
  "abstention_status_accuracy": 1.0,
  "note": "Amostra autoral pequena. As métricas não avaliam a correção da resposta gerada. Rótulos sujeitos a revisão independente."
}
{
  "date": "2026-11-25",
  "split": "test",
  "k": 3,
  "cases": 10,
  "answerable_cases": 6,
  "abstention_cases": 4,
  "macro_precision_at_3": 0.3333,
  "macro_precision_returned": 0.6667,
  "macro_recall_at_3": 1.0,
  "macro_f1_at_3": 0.5,
  "hit_at_3": 1.0,
  "hit_at_1": 1.0,
  "mrr_at_3": 1.0,
  "answerable_status_accuracy": 1.0,
  "abstention_status_accuracy": 1.0,
  "note": "Amostra autoral pequena. As métricas não avaliam a correção da resposta gerada. Rótulos sujeitos a revisã

{
  "date": "2026-11-25",
  "split": "parafrase",
  "k": 3,
  "cases": 20,
  "answerable_cases": 17,
  "abstention_cases": 3,
  "macro_precision_at_3": 0.3333,
  "macro_precision_returned": 0.451,
  "macro_recall_at_3": 0.9412,
  "macro_f1_at_3": 0.4882,
  "hit_at_3": 0.9412,
  "hit_at_1": 0.7647,
  "mrr_at_3": 0.8431,
  "answerable_status_accuracy": 0.9412,
  "abstention_status_accuracy": 0.0,
  "note": "Amostra autoral pequena. As métricas não avaliam a correção da resposta gerada. Rótulos sujeitos a revisão independente."
}

A amostra é pequena e foi elaborada para o recorte deste protótipo. Para uma avaliação mais ampla, reúna novas perguntas de estudantes, peça revisão dos rótulos e avalie separadamente a exatidão, a completude e o apoio das fontes nas respostas geradas.